# Recipe Prediction Analysis

**Name(s)**: Coleman Clougherty and Jamera Fernando

**Website Link**: (your website link)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

from dsc80_utils import * # Feel free to uncomment and use this.
import re
import matplotlib.pyplot as plt
from tqdm import trange
from sklearn.linear_model import LinearRegression

## Step 1: Introduction

Our data comes from food.com, which is a platform where users can publically submit personalized recipes to learn from, rate, and review them. One of the features of food.com is that each recipe has different tags that describe the kind of dishes they are (e.g. 'salsas', 'nut-free', 'french'). Given this, we wanted to investigate the question: **what is the relationship between features, such as cooking time, ingredients, and calories, with country of origin for different recipes?**


## Step 2: Data Cleaning and Exploratory Data Analysis

In [3]:
interactions = pd.read_csv(Path('data') /'RAW_interactions.csv')
recipes = pd.read_csv(Path('data') / 'RAW_recipes.csv')

In [4]:
recipes['nutrition']

0            [138.4, 10.0, 50.0, 3.0, 3.0, 19.0, 6.0]
1        [595.1, 46.0, 211.0, 22.0, 13.0, 51.0, 26.0]
2           [194.8, 20.0, 6.0, 32.0, 22.0, 36.0, 3.0]
                             ...                     
83779            [59.2, 6.0, 2.0, 3.0, 6.0, 5.0, 0.0]
83780       [188.0, 11.0, 57.0, 11.0, 7.0, 21.0, 9.0]
83781        [174.9, 14.0, 33.0, 4.0, 4.0, 11.0, 6.0]
Name: nutrition, Length: 83782, dtype: object

In [5]:
recipes['tags']

0        ['60-minutes-or-less', 'time-to-make', 'course...
1        ['60-minutes-or-less', 'time-to-make', 'cuisin...
2        ['60-minutes-or-less', 'time-to-make', 'course...
                               ...                        
83779    ['60-minutes-or-less', 'time-to-make', 'course...
83780    ['30-minutes-or-less', 'time-to-make', 'course...
83781    ['30-minutes-or-less', 'time-to-make', 'course...
Name: tags, Length: 83782, dtype: object

In [6]:
recipes.columns

Index(['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'ingredients',
       'n_ingredients'],
      dtype='object')

In [7]:
interactions['recipe_id'].sort_values()

70222         38
70223         38
70221         38
           ...  
475342    537543
277439    537671
295955    537716
Name: recipe_id, Length: 731927, dtype: int64

In [8]:
interactions[interactions['review'].isna()].describe()

,user_id,recipe_id,rating
count,1.69e+02,169.00,169.00
mean,1.83e+09,214467.93,4.72
std,5.52e+08,163237.62,0.70
...,...,...,...
50%,2.00e+09,166775.00,5.00
75%,2.00e+09,356724.00,5.00
max,2.00e+09,533896.00,5.00


We created our merged recipes data sets and replaced average ratings that had 0 with np.nan

In [9]:
merged = recipes.merge(interactions, left_on='id', right_on='recipe_id', how='left')
merged.loc[merged['rating'] == 0, 'rating'] = np.nan
avg_rating = merged.groupby('id')['rating'].mean()
avg_rating

recipes2 = recipes.merge(avg_rating.rename('avg_rating'), left_on='id', right_index=True, how='left')
# recipes[recipes['avg_rating'].notnull()]
recipes2

,name,id,minutes,contributor_id,...,description,ingredients,n_ingredients,avg_rating
0,1 brownies in the world best ever,333281,40,985201,...,"these are the most; chocolatey, moist, rich, d...","['bittersweet chocolate', 'unsalted butter', '...",9,4.0
1,1 in canada chocolate chip cookies,453467,45,1848091,...,this is the recipe that we use at my school ca...,"['white sugar', 'brown sugar', 'salt', 'margar...",11,5.0
2,412 broccoli casserole,306168,40,50969,...,since there are already 411 recipes for brocco...,"['frozen broccoli cuts', 'cream of chicken sou...",9,5.0
...,...,...,...,...,...,...,...,...,...
83779,zydeco ya ya deviled eggs,308080,40,37779,...,"deviled eggs, cajun-style","['hard-cooked eggs', 'mayonnaise', 'dijon must...",8,5.0
83780,cookies by design cookies on a stick,298512,29,506822,...,"i've heard of the 'cookies by design' company,...","['butter', 'eagle brand condensed milk', 'ligh...",10,1.0
83781,cookies by design sugar shortbread cookies,298509,20,506822,...,"i've heard of the 'cookies by design' company,...","['granulated sugar', 'shortening', 'eggs', 'fl...",7,3.0


In [10]:
recipes2['tags'].iloc[0]

"['60-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'preparation', 'for-large-groups', 'desserts', 'lunch', 'snacks', 'cookies-and-brownies', 'chocolate', 'bar-cookies', 'brownies', 'number-of-servings']"

The tags column is a quote of a list of different tokens, so we wanted to canonicalize it and get each separate token. We then created a new column, 'tags_list' which transformed tags (quote) into a list of individually quoted tags.

In [11]:
def tags_to_list(tags):
    pattern =r"'([\w-]*)'"
    unused = r"('(.*)',)|(, '(.*)')"
    return re.findall(pattern, tags)

recipes2['tags_list'] = recipes2['tags'].transform(tags_to_list)

# nutrition_dict={}
# def nutrition_to_columns(nutrition):
#     pattern = r"\d+\.\d*"
#     matches = re.findall(pattern, nutrition)
#     nutrition_dict['calories'] = nutrition_dict.get('calories', []) + [float(matches[0])]
#     nutrition_dict['total_fat'] = nutrition_dict.get('fat', []) + [float(matches[1])]
#     nutrition_dict['sugar'] = nutrition_dict.get('sugar', []) + [float(matches[2])]
#     nutrition_dict['sodium'] = nutrition_dict.get('sodium', []) + [float(matches[3])]
#     nutrition_dict['protein'] = nutrition_dict.get('protein', []) + [float(matches[4])]
#     nutrition_dict['saturated_fat'] = nutrition_dict.get('saturated_fat', []) + [float(matches[5])]
#     nutrition_dict['carbohydrates'] = nutrition_dict.get('carbohydrates', []) + [float(matches[6])]

df_extracted = recipes2['nutrition'].str.findall(r"\d+\.\d*").apply(pd.Series)
df_extracted.columns = ['calories', 'total_fat', 'sugar', 'sodium', 'protein', 'saturated_fat', 'carbohydrates']
df_extracted

recipes3 = pd.concat([recipes2, df_extracted], axis=1).drop(columns=['nutrition', 'tags'])
recipes3['tags_list'].iloc[0]

['60-minutes-or-less',
 'time-to-make',
 'course',
 'main-ingredient',
 'preparation',
 'for-large-groups',
 'desserts',
 'lunch',
 'snacks',
 'cookies-and-brownies',
 'chocolate',
 'bar-cookies',
 'brownies',
 'number-of-servings']

In [11]:
recipes3['tags_list'].iloc[501]

['30-minutes-or-less',
 'time-to-make',
 'course',
 'main-ingredient',
 'cuisine',
 'preparation',
 'north-american',
 'main-dish',
 'beans',
 'vegetables',
 'mexican',
 'easy',
 'beginner-cook',
 'kid-friendly',
 'vegetarian',
 'dietary',
 'one-dish-meal',
 'inexpensive']

Since we wanted to investigate ingredients to see if there is a relationship 

In [21]:
recipes3['ingredients'].transform(tags_to_list)

0                                          [eggs]
1         [salt, margarine, eggs, vanilla, water]
2                                    [salt, milk]
                           ...                   
83779                          [mayonnaise, salt]
83780    [butter, egg, extract, nutmeg, bisquick]
83781                   [shortening, eggs, flour]
Name: ingredients, Length: 83782, dtype: object

We first decided to do country tags to categorize the categories. However, since there were too many countries and not enough rows, we decided on using continent tags to label the recipes.

In [20]:
country_tags = ['american',
                'mexican',
                'french',
                'german',
                'italian',
                'indian',
                'portuguese',
                'japanese',
                'thai',
                'spanish',
                'greek',
                'chinese',
                'vietnamese',
                'turkish',
                'brazilian',
                'lebanese',
                'russian',
                'moroccan',
                'south-african',
                'canadian',
                'iranian-persian',
                'swedish',
                'australian',
                'swiss',
                'scottish',
                'irish'
                ]
continent_tags = [
    'north-american',
    'south-american',
    'european',
    'asian',
    'african',
    'australian'
]


In [22]:
all_tags = [str(tag) for sublist in recipes2['tags_list'] for tag in sublist]
unique_tags = list(set(all_tags))

In [23]:
def tag_contains(tags, name):
    name = name.lower()
    return name in tags

In [24]:
for continent in continent_tags:
    print(continent, recipes3['tags_list'].apply(tag_contains, name=continent).sum())

north-american 14590
south-american 689
european 8897
asian 4575
african 1357
australian 1151


### Distribution of average_ratings in Recipes ###

In [25]:
recipes3['avg_rating'].hist(nbins=10)

As seen in the graph above, the distribution is unimodal with a strong skew to the left. Higher rated recipes usually have the highest amount of reviews; this could be because popular recipes, made by popular chefs for example, are more critically acclaimed and praised. In contrast, recipes with low views, specifically below 10k, have ratings that are average to poor. 

In [26]:
px.histogram(recipes3, x='avg_rating', nbins=10, title='Average Recipe Ratings', labels={'avg_rating':'Average Rating', 'count':'Count'})

Average recipe ratings remain the same distribution as the raw distribution of recipe ratings.

In [27]:
data = interactions.groupby('recipe_id')['rating'].count()

In [28]:
px.box(data, x='rating', title='Number of Ratings per Recipe', log_x=True, labels = {'rating':'Number of Ratings (scaled log)'})

In [29]:
recipes3['tags_list']

0        [60-minutes-or-less, time-to-make, course, mai...
1        [60-minutes-or-less, time-to-make, cuisine, pr...
2        [60-minutes-or-less, time-to-make, course, mai...
                               ...                        
83779    [60-minutes-or-less, time-to-make, course, mai...
83780    [30-minutes-or-less, time-to-make, course, pre...
83781    [30-minutes-or-less, time-to-make, course, pre...
Name: tags_list, Length: 83782, dtype: object

In [30]:
all_tags = [str(tag) for sublist in recipes2['tags_list'] for tag in sublist]
unique_tags = list(set(all_tags))

In [31]:
all_tags

['60-minutes-or-less',
 'time-to-make',
 'course',
 'main-ingredient',
 'preparation',
 'for-large-groups',
 'desserts',
 'lunch',
 'snacks',
 'cookies-and-brownies',
 'chocolate',
 'bar-cookies',
 'brownies',
 'number-of-servings',
 '60-minutes-or-less',
 'time-to-make',
 'cuisine',
 'preparation',
 'north-american',
 'for-large-groups',
 'canadian',
 'british-columbian',
 'number-of-servings',
 '60-minutes-or-less',
 'time-to-make',
 'course',
 'main-ingredient',
 'preparation',
 'side-dishes',
 'vegetables',
 'easy',
 'beginner-cook',
 'broccoli',
 'time-to-make',
 'course',
 'cuisine',
 'preparation',
 'occasion',
 'north-american',
 'desserts',
 'american',
 'southern-united-states',
 'dinner-party',
 'holiday-event',
 'cakes',
 'dietary',
 'christmas',
 'thanksgiving',
 'low-sodium',
 'low-in-something',
 'taste-mood',
 'sweet',
 '4-hours-or-less',
 'time-to-make',
 'course',
 'main-ingredient',
 'preparation',
 'main-dish',
 'potatoes',
 'vegetables',
 '4-hours-or-less',
 'meatl

In [32]:
unique_tags

['',
 'potatoes',
 'thanksgiving',
 'peanut-butter',
 'libyan',
 'low-fat',
 'berries',
 'pasta-salad',
 'tropical-fruit',
 'superbowl',
 'black-bean-soup',
 'chilean',
 'duck',
 'onions',
 'leftovers',
 'scallops',
 'indonesian',
 'side-dishes',
 'herb-and-spice-mixes',
 'cuban',
 'small-appliance',
 'savory',
 'passover',
 'pork-ribs',
 'comfort-food',
 'swedish',
 'pork-chops',
 'high-calcium',
 'cinco-de-mayo',
 'chocolate-chip-cookies',
 'oatmeal',
 'oaxacan',
 'cabbage',
 'healthy',
 'pies',
 'turkish',
 'whole-chicken',
 'blueberries',
 'squid',
 'jewish-sephardi',
 'baking',
 'moose',
 'fruit',
 'chicken-stews',
 'duck-breasts',
 'vegetarian',
 'peruvian',
 'snacks-kid-friendly',
 'pot-pie',
 'new-zealand',
 'low-in-something',
 'food-processor-blender',
 'pears',
 'toddler-friendly',
 'breads',
 'low-protein',
 'midwestern',
 'manicotti',
 'served-cold',
 'ethiopian',
 'freshwater-fish',
 'soy-tofu',
 'spaghetti-sauce',
 'mango',
 'pork',
 'ramadan',
 'shrimp',
 'mixer',
 'mic

In [33]:
recipes3['name'] = recipes3['name'].fillna('')

In [34]:
ser_tags = pd.Series(unique_tags)

In [35]:
ser_tags[ser_tags.str.contains('india')]

535    indian
dtype: object

In [36]:
'food' in unique_tags

False

In [37]:
def tag_contains(tags, name):
    name = name.lower()
    return name in tags

christmas = recipes3[recipes3['tags_list'].apply(tag_contains, name='halloween')]
christmas[christmas['tags_list'].apply(tag_contains, name='holiday-event')].shape[0]
# christmas.shape[0]

229

In [38]:
american = recipes3.copy()
american['is_american'] = recipes3['tags_list'].apply(tag_contains, name='american')


In [39]:
american[american['is_american']]

,name,id,minutes,contributor_id,...,protein,saturated_fat,carbohydrates,is_american
3,millionaire pound cake,286009,120,461724,...,20.0,123.0,39.0,True
11,rter med flsk pea soup with pork,333797,195,64642,...,44.0,12.0,0.0,True
15,go to bbq sauce for ribs,495314,13,488441,...,2.0,0.0,19.0,True
...,...,...,...,...,...,...,...,...,...
83759,zuppa di pesce castagna,392620,70,62264,...,68.0,15.0,6.0,True
83776,zydeco sauce,357451,15,461283,...,1.0,14.0,5.0,True
83777,zydeco soup,486161,60,227978,...,44.0,21.0,15.0,True


In [40]:
american['calories'] = american['calories'].astype(float)
american['total_fat'] = american['total_fat'].astype(float)

In [41]:
american['total_fat'].describe()

count    83782.00
mean        32.63
std         60.15
           ...   
50%         20.00
75%         39.00
max       3464.00
Name: total_fat, Length: 8, dtype: float64

In [42]:
american['calories_from_fat'] = american['total_fat'] * 0.78 * 9

In [43]:
american['percent_calories_from_fat'] = american['calories_from_fat'] / american['calories'] * 100

In [44]:
american['percent_calories_from_fat'].hist(nbins=20)

In [45]:
american

,name,id,minutes,contributor_id,...,carbohydrates,is_american,calories_from_fat,percent_calories_from_fat
0,1 brownies in the world best ever,333281,40,985201,...,6.0,False,70.20,50.72
1,1 in canada chocolate chip cookies,453467,45,1848091,...,26.0,False,322.92,54.26
2,412 broccoli casserole,306168,40,50969,...,3.0,False,140.40,72.07
...,...,...,...,...,...,...,...,...,...
83779,zydeco ya ya deviled eggs,308080,40,37779,...,0.0,False,42.12,71.15
83780,cookies by design cookies on a stick,298512,29,506822,...,9.0,False,77.22,41.07
83781,cookies by design sugar shortbread cookies,298509,20,506822,...,6.0,False,98.28,56.19


In [46]:
american.pivot_table(values='calories', index='is_american', aggfunc='mean')

,calories
is_american,
False,427.02
True,453.37


In [47]:
american[american['is_american']]['name']


3                    millionaire pound cake
11       rter med flsk   pea soup with pork
15                 go to bbq sauce for ribs
                        ...                
83759               zuppa di pesce castagna
83776                          zydeco sauce
83777                           zydeco soup
Name: name, Length: 9257, dtype: object

In [48]:
# print(american.head())
fig = px.histogram(american,
             x='percent_calories_from_fat',
             nbins = 20, 
             color='is_american', 
             barmode='overlay', 
             histnorm='probability density', 
             title='Distribution of Approximate Percent Calories from Fat<br>in American Recipes vs. Non-American Recipes', 
             labels={'percent_calories_from_fat':'Percent Calories from Fat', 'probability density':'Probability Density', 'is_american':'Is American Recipe?'},
             width=900,
             height=500)
fig.update_xaxes(dtick=10)
fig.show()

In [49]:
recipes3[recipes3['name'].str.lower().str.contains('filipino')]

,name,id,minutes,contributor_id,...,sodium,protein,saturated_fat,carbohydrates
1024,adobo style shrimp filipino,374461,11,544754,...,201.0,97.0,12.0,4.0
11344,buko salad filipino young coconut fruit salad,306016,10,385678,...,3.0,12.0,86.0,17.0
12401,camaron rebosado filipino fried shrimp,527455,42,314579,...,52.0,40.0,110.0,5.0
...,...,...,...,...,...,...,...,...,...
76570,the filipino elvis sandwich,527485,13,135470,...,36.0,12.0,221.0,18.0
80329,vegetarian pansit noodles filipino,305777,50,385678,...,26.0,16.0,14.0,20.0
80358,vegetarian sinigang filipino tamarind or sour...,313263,35,602448,...,7.0,29.0,3.0,11.0


In [50]:
recipes3['tags_list'].count()

np.int64(83782)

In [51]:
recipes3['ingredients'].iloc[0]

"['bittersweet chocolate', 'unsalted butter', 'eggs', 'granulated sugar', 'unsweetened cocoa powder', 'vanilla extract', 'brewed espresso', 'kosher salt', 'all-purpose flour']"

In [52]:
# all_tags = recipes2['tags_list'].sum()

In [53]:
# TODO

## Step 3: Assessment of Missingness

In [54]:
# TODO

## Step 4: Hypothesis Testing

**Permutation Test #1**
- Null Hypothesis: There is no difference in means for calorie count in American and non-American recipes.
- Alternative Hypothesis: American recipes have a higher mean calorie count than non-American recipes.

In [55]:
def calculate_diff(df):
    pivot = df.pivot_table(values='calories', index='is_american', aggfunc='mean')
    return pivot.loc[True, 'calories'] - pivot.loc[False, 'calories']

In [56]:
observed_diff = calculate_diff(american)
observed_diff

np.float64(26.35279750080423)

In [57]:
from tqdm import trange

In [58]:
repititions = 1000
diffs = []
for _ in trange(repititions):
    shuffled = american.assign(is_american = np.random.permutation(american['is_american']))
    diff = calculate_diff(shuffled)
    diffs.append(diff)
px.histogram(diffs, nbins=30, title='Null Distribution of Difference in Mean Calories<br>Between American and Non-American Recipes', labels={'value':'Difference in Mean Calories', 'count':'Count'}, width=900, height=500).show()

  0%|          | 0/1000 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [01:37<00:00, 10.28it/s]


In [59]:
fig = px.histogram(diffs, x=0, nbins=30, 
                   title='Null Distribution of Difference in Mean Calories<br>Between American and Non-American Recipes', 
                   labels={'value':'Difference in Mean Calories', 'count':'Count'}, 
                   width=1000, height=500)
fig.add_vline(x=observed_diff, line_color='red', opacity=1, line_width=3).show()
fig.add_annotation(text=f'<span style="color:red">Observed TVD', showarrow=False, y=0.16)
fig.update_layout(yaxis_range=[0, 0.2])

In [60]:
diffs_ser = pd.Series(diffs)
p_value = (diffs_ser >= observed_diff).sum() / repititions
p_value

np.float64(0.001)

**Permutation Test #2**
- Null Hypothesis: There is no correlation between cooking time and star ratings. 
- Alternative Hypothesis: There is a positive correlation between cooking time and star ratings.


In [61]:
french=recipes3.copy()
french['is_french'] = recipes3['tags_list'].apply(tag_contains, name='french')


In [62]:
ser_tags[ser_tags.str.contains('cheese')]

140                 cheese
176             cheesecake
539    macaroni-and-cheese
dtype: object

In [63]:
px.scatter(recipes3, x='avg_rating', y='sugar')

In [64]:
recipes3['tags_list'].apply(tag_contains, name='french').sum()

np.int64(803)

In [65]:
ratings = recipes3[['avg_rating', 'sugar']].dropna()
ratings

,avg_rating,sugar
0,4.0,50.0
1,5.0,211.0
2,5.0,6.0
...,...,...
83779,5.0,2.0
83780,1.0,57.0
83781,3.0,33.0


In [66]:
model = LinearRegression()
model.fit(ratings[['sugar']], ratings['avg_rating'])

LinearRegression()

In [67]:
def calculate_R(df):
    model = LinearRegression()
    model.fit(df[['sugar']], df['avg_rating'])
    return model.score(df[['sugar']], df['avg_rating'])

In [68]:
observed_R = calculate_R(ratings)

In [69]:
repititions = 1000
R_scores = []
for _ in trange(repititions):
    shuffled = ratings.assign(avg_rating = np.random.permutation(ratings['avg_rating']))
    R = calculate_R(shuffled)
    R_scores.append(R)

100%|██████████| 1000/1000 [01:02<00:00, 15.96it/s]


In [70]:
fig = px.histogram(R_scores, nbins=30, title='Null Distribution of R-squared<br>Between Sugar and Average Rating', labels={'value':'R-squared', 'count':'Count'}, width=900, height=500)
fig.add_vline(x=observed_R, line_color='red', opacity=1, line_width=3).show()

p_value = (pd.Series(R_scores) >= observed_R).sum() / repititions
p_value

np.float64(0.806)

In [71]:
model.score(ratings[['sugar']], ratings['avg_rating'])

6.627097844935648e-07

## Step 5: Framing a Prediction Problem

In [72]:
# TODO

## Step 6: Baseline Model

In [73]:
# TODO

## Step 7: Final Model

In [74]:
# TODO

## Step 8: Fairness Analysis

In [75]:
# TODO